# 03 - Transform Wuzzuf Jobs

Stage 3 of the pipeline.

1. Reads **only** `PENDING_CLEANED_QUEUE` — the batch notebook 2 cleaned since this notebook last ran. If it's empty, there's nothing new to transform and the notebook stops after saying so.
2. Adds `source`, `collected_at`, and a placeholder `estimated_salary`, and enforces the final column order.
3. Appends the transformed batch onto the final history (`FINAL_JSON`), deduplicated by `job_url`, and rewrites `FINAL_CSV` from that full history so the CSV is always a consistent, duplicate-free snapshot.
4. Clears `PENDING_CLEANED_QUEUE` now that it's been consumed.

In [ ]:
import csv
import json
import logging
import os
from datetime import datetime, timezone

import pandas as pd

## Settings

In [ ]:
PENDING_CLEANED_QUEUE = "../data/interim/wuzzuf_pending_cleaned_queue.json"  # input: unconsumed batch from notebook 2
FINAL_JSON = "../data/processed/wuzzuf_jobs_final.json"    # output: full final history (audit)
FINAL_CSV = "../data/processed/wuzzuf_jobs_final.csv"      # output: full history as CSV

FINAL_SCHEMA = [
    "job_id",
    "source",
    "job_title",
    "company_name",
    "job_description",
    "job_url",
    "country",
    "city",
    "work_mode",
    "employment_type",
    "career_level",
    "experience_needed",
    "qualification_required",
    "skills",
    "salary",
    "estimated_salary",
    "posted_date",
    "collected_at",
]

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("wuzzuf.transform")

## Queue / history helpers

In [ ]:
def load_json(path):
    if not os.path.exists(path):
        return []
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(records, path):
    dirname = os.path.dirname(path)
    if dirname:
        os.makedirs(dirname, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)


def dedupe_keep_last(records, key="job_url"):
    """Merge records, keeping the most recent version of each key."""
    deduped = {}
    for r in records:
        k = r.get(key)
        if k is None:
            continue
        deduped[k] = r
    return list(deduped.values())

## Load the pending batch

In [ ]:
pending_cleaned = load_json(PENDING_CLEANED_QUEUE)
log.info("Pending cleaned queue: %d job(s) to transform", len(pending_cleaned))

## Build the final record (schema + derived fields)

In [ ]:
def transform_record(record):
    """Add pipeline metadata fields and enforce the final column order."""
    collected_at = datetime.now(timezone.utc).replace(tzinfo=None).strftime("%Y-%m-%d %H:%M:%S")

    final = {
        **record,
        "source": "Wuzzuf",
        # Wuzzuf doesn't expose a real estimated salary.
        # Left as None here for a later ML/estimation step to populate.
        "estimated_salary": None,
        "collected_at": collected_at,
    }

    return {column: final.get(column) for column in FINAL_SCHEMA}

## CSV helpers

In [ ]:
def _stringify_csv_value(value):
    if value is None:
        return ""

    # Skills: ["Python", "SQL", "Snowflake"] -> "Python; SQL; Snowflake"
    if isinstance(value, list):
        return "; ".join(str(item) for item in value)

    if isinstance(value, dict):
        return json.dumps(value, ensure_ascii=False)

    return value


def save_csv(records, filename, fieldnames):
    dirname = os.path.dirname(filename)
    if dirname:
        os.makedirs(dirname, exist_ok=True)

    with open(filename, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        for record in records:
            writer.writerow({c: _stringify_csv_value(record.get(c)) for c in fieldnames})

## Transform the pending batch, update history

A no-op (with a log message) when `pending_cleaned` is empty, so it's always safe to run this notebook even when notebook 2 hasn't produced anything new.

In [ ]:
if not pending_cleaned:
    log.info("Nothing pending. Skipping transform.")
    final_history = load_json(FINAL_JSON)
else:
    transformed_batch = [transform_record(r) for r in pending_cleaned]

    final_history = load_json(FINAL_JSON)
    final_history = dedupe_keep_last(final_history + transformed_batch, key="job_url")
    save_json(final_history, FINAL_JSON)

    # Rewrite the CSV from the full (deduplicated) history so it's always consistent
    save_csv(final_history, FINAL_CSV, FINAL_SCHEMA)

    # This batch is consumed - clear the cleaned queue
    save_json([], PENDING_CLEANED_QUEUE)

    log.info("Transformed %d job(s) | Final history total: %d | Cleaned queue cleared",
              len(transformed_batch), len(final_history))

len(final_history)

## Quick preview

In [ ]:
df = pd.DataFrame(final_history, columns=FINAL_SCHEMA)
df.head()